In [106]:
from sqlalchemy import create_engine, text
import pandas as pd
import numpy as np
from datetime import datetime

import credentials

In [224]:
conn_str = 'mysql+mysqlconnector://' + \
	f"{credentials.mysql['username']}:{credentials.mysql['password']}" + \
	'@localhost:3306/Mutis'
engine = create_engine(conn_str)
connection = engine.connect()

In [119]:
colls = pd.read_csv("collectors.csv", delimiter="\t")
colls["Standarized_Collector"] = colls.Standarized_Collector.str.replace(".", ". "
	).str.replace(",", ", "
	).str.replace(r"\.\s+,", ".,", regex=True
	).str.replace(r"\s+", " ", regex=True
	).str.replace(r"\s+$|^\s+", "", regex=True
	)

In [120]:
persons = pd.read_sql_table("Persons", connection)

In [121]:
qu = ~colls.Standarized_Collector.isin(persons.NameVerbatim)
colls[["LastName", "FirstName"]] = colls.Standarized_Collector.str.split(
	r",\s+", regex=True, expand=True
	)
colls = colls[qu].reset_index(drop=True)
startidx = persons.PersonID.max() + 1
colls["PersonID"] = colls.index + startidx
colls[["Abbreviation", "Affiliation", "Email"]] = None, None, None
colls = colls.rename(columns={"Standarized_Collector":"NameVerbatim"})
colls["TimeStamp"] = datetime.strptime("2026-02-16", "%Y-%m-%d")

In [124]:
colls[persons.columns].to_sql('Persons', engine, if_exists='append', index=False, 
		chunksize=10000, method='multi'
	)

-1

In [228]:
persons = pd.read_sql_table("Persons", connection)

In [229]:
peoper = pd.read_sql_table("PeoplePersons", connection)

In [230]:
people = pd.read_sql_table("People", connection)

In [225]:
maxper = startidx - 1
maxpeople = people.PeopleID.max()

In [226]:
maxper, maxpeople

(170, 153)

In [227]:
# The following works because all collectors are one-person teams
peopleid = maxpeople

for perid in persons.PersonID[persons.PersonID > maxper]:
	peopleid += 1

	s = text("INSERT INTO People(PeopleID) VALUES (:ppid)")
	connection.execute(s, {"ppid": peopleid})
	connection.commit()

	s = text("INSERT INTO PeoplePersons VALUES (:ppid, :perid, 1, '2026-02-16')")
	connection.execute(s, {"ppid": peopleid, "perid": perid})
	connection.commit()


In [234]:
occcolls = pd.read_sql_query("SELECT CollectorVerbatim, Collector FROM Occurrences WHERE CollectorVerbatim IS NOT NULL", connection)

In [238]:
colls

,CollectorVerbatim,NameVerbatim,LastName,FirstName,PersonID,Abbreviation,Affiliation,Email,TimeStamp
0,"Acero, E.","Acero, Enrique",Acero,Enrique,171,None,None,None,2026-02-16
1,"Acero, J.D.","Acero, J. D.",Acero,J. D.,172,None,None,None,2026-02-16
2,"Acevedo-Luna, N.I.","Acevedo-Luna, N. I.",Acevedo-Luna,N. I.,173,None,None,None,2026-02-16
3,"Acevedo, E. de","Acevedo, E. de",Acevedo,E. de,174,None,None,None,2026-02-16
4,"Acevedo, O.","Acevedo, O.",Acevedo,O.,175,None,None,None,2026-02-16
...,...,...,...,...,...,...,...,...,...
1325,"Zucolotto, S.","Zucolotto, S.",Zucolotto,S.,1496,None,None,None,2026-02-16
1326,"Zuluaga-C., Juliana","Zuluaga-C., Juliana",Zuluaga-C.,Juliana,1497,None,None,None,2026-02-16
1327,"Zuluaga-R., S.","Zuluaga-R., S.",Zuluaga-R.,S.,1498,None,None,None,2026-02-16
1328,"Zulueta, I. de","Zulueta, I. de",Zulueta,I. de,1499,None,None,None,2026-02-16


In [237]:
persons[persons.NameVerbatim.str.startswith('Acero')]

,PersonID,NameVerbatim,FirstName,LastName,Abbreviation,Affiliation,Email,TimeStamp
170,171,"Acero, Enrique",Enrique,Acero,None,None,None,2026-02-16
171,172,"Acero, J. D.",J. D.,Acero,None,None,None,2026-02-16


In [232]:
people

,PeopleID,TimeStamp
0,1,2025-06-18 09:15:52
1,2,2025-06-18 09:15:52
2,3,2025-06-18 09:16:06
3,4,2025-06-18 09:16:06
4,5,2025-06-18 09:16:06
...,...,...
1478,1479,2026-02-16 20:17:50
1479,1480,2026-02-16 20:17:50
1480,1481,2026-02-16 20:17:50
1481,1482,2026-02-16 20:17:50


In [246]:
colls[colls.NameVerbatim == "Lancheros, Héctor Orlando"]

,CollectorVerbatim,NameVerbatim,LastName,FirstName,PersonID,Abbreviation,Affiliation,Email,TimeStamp
677,"Lancheros, H.","Lancheros, Héctor Orlando",Lancheros,Héctor Orlando,848,None,None,None,2026-02-16
678,"Lancheros, Héctor O.","Lancheros, Héctor Orlando",Lancheros,Héctor Orlando,849,None,None,None,2026-02-16
679,"Lancheros, Héctor Orlando","Lancheros, Héctor Orlando",Lancheros,Héctor Orlando,850,None,None,None,2026-02-16


In [244]:
for row in occcolls.itertuples():

	if pd.isna(row.Collector):

		try:

			stdcoll = colls.loc[
				colls.CollectorVerbatim == row.CollectorVerbatim,
				"NameVerbatim"].item()


			perid = persons.loc[
				persons.NameVerbatim == stdcoll, 
				"PersonID"
				].item()
			
		except:
			#print(f"{perid=}")
			print(f"{row.CollectorVerbatim=}")
			print(f"{stdcoll=}")
			continue

		peopleid = peoper.loc[
			(peoper.Person == perid),# & (peoper.Order == 1), 
			"People"].item()

		#print(f"{perid=}, {peopleid=}")

		s = text("SELECT OccurrenceID FROM Occurrences where CollectorVerbatim = :thcoll")
		thoccs = connection.execute(s, {"thcoll": row.CollectorVerbatim}).fetchall()

		#print(f"{thoccs}")

		#for i in (x[0] for x in thoccs):

		#	s = text("UPDATE Occurrences SET Collector = :ppid WHERE OccurrenceID = :occid")
		#	connection.execute(s, {"ppid": peopleid, "occid": i})
		#	connection.commit()

		#break


row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumn

KeyboardInterrupt: 

In [149]:
colls.loc[colls.NameVerbatim == collector, "CollectorVerbatim"].item()

'Acero, E.'

In [223]:
connection.close()